# Chopper-Frequency Sweep Demo

Sweeps the QDac CH1 on/off chopper frequency (holding CH2's fast pulse
trigger frequency, the Avtech lasing voltage, and the MFLI's demodulator
filter/rate fixed) to find which chopper frequency gives the best lock-in
SNR, via `measurement.chopper_sweep.run_chopper_frequency_sweep`.

CH1 and CH2 share the same QDac trigger group so they stay phase-locked
with no timing jitter between them -- every step resets *both* channels
together (abort, reconfigure, re-arm, fire) rather than touching CH1
alone. CH1's candidate frequencies are restricted to common factors of
CH2's frequency, so the pulse train stays stable relative to the chopper.

This is a single-cell "run and save" workflow, same pattern as
`05_full_sweep_demo.ipynb`:

- **Live plotting**: `on_step` drives a `clear_output(wait=True)`-based
  live-updating R-vs-frequency plot (log frequency axis) as the sweep
  progresses.
- **Interrupt safety**: interrupting the sweep cell is caught internally
  by `run_chopper_frequency_sweep`. Either way -- completed, interrupted,
  or timed out (`SweepConfig.max_runtime_s`, default 10000 s given the
  longer settle times at low chopper frequencies) -- the Avtech is always
  ramped to 0 V and disabled, and CH1 is always reset back to its default
  200 Hz (CH2 unchanged), before the function returns.
- **Always a result**: the same cell always ends with a saved
  `sweep.csv` + `metadata.json` pair (including the fixed MFLI
  filter/rate settings) in a timestamped folder under `data/`.

Rigol is not used in this experiment.

**No real instruments are connected to this development machine.** If any
instrument fails to connect, this notebook falls back to clearly-labeled
synthetic `dummy_data` so the saving/plotting logic can still be
demonstrated end-to-end. On the lab computer, with QDac/Avtech/MFLI
connected, the same cells should run the real sweep instead.

In [ ]:
import sys
from pathlib import Path

# Make sure the project root (containing `instruments/`) is importable,
# whether this notebook is run from the repo root or from within notebooks/.
project_root = Path.cwd()
if not (project_root / "instruments").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output

from instruments.qdac import QDac
from instruments.avtech import Avtech
from instruments.mfli import MFLI
from instruments import config
from measurement.chopper_sweep import (
    run_chopper_frequency_sweep,
    ChopperSweepConfig,
    ChopperSweepResult,
)
from measurement import data_io, plotting

## Instantiate and configure QDac, Avtech, and MFLI (no Rigol)

In [ ]:
qdac = QDac()
avtech = Avtech()
mfli = MFLI()

instruments_connected = False

try:
    # QDac: CH1 is the on/off chopper (default 200 Hz here), CH2 is the
    # fast pulse trigger (2000 Hz). Both share trigger group INT1 so they
    # start in lockstep with no jitter between them.
    qdac.connect()
    gate_voltage = 5.0
    qdac.channel_init(1)
    qdac.channel_init(2)
    qdac.abort_square_wave(1)
    qdac.abort_square_wave(2)
    qdac.configure_square_wave(
        1, frequency=200, span=gate_voltage, offset=gate_voltage / 2, trigger_source=1
    )
    qdac.configure_square_wave(
        2,
        frequency=2000,
        span=gate_voltage,
        offset=gate_voltage / 2,
        trigger_source=1,
        delay=0.0125e-3,
        duty_cycle=50,
    )
    qdac.start_square_wave(1)
    qdac.start_square_wave(2)
    qdac.fire_internal_trigger(1)

    # Avtech: pulse voltage held fixed at the lasing voltage throughout,
    # externally triggered by the QDac.
    avtech.connect()
    avtech.remote()
    avtech.set_trigger("EXT")
    avtech.set_pulse_width(80e-6)
    avtech.output_on()

    # MFLI: reads the lock-in signal corresponding to optical intensity.
    # Demodulator filter/rate are set once per sweep by
    # run_chopper_frequency_sweep itself (kept fixed across all chopper
    # frequencies); this just applies the base input/reference config.
    mfli.connect()
    mfli.apply_default_configuration()

    instruments_connected = True
    print("QDac, Avtech, and MFLI connected and configured.")
except Exception as exc:
    print(f"Hardware not available: {exc}")
    print("Falling back to synthetic dummy_data later in this notebook.")

## Chopper frequencies

CH1's candidate frequencies are all common factors of CH2's 2000 Hz (up
to 1000 Hz), so the pulse train stays stable relative to the chopper.

In [ ]:
chopper_frequencies_hz = [f for f in range(1, 1001) if 2000 % f == 0]
print(f"{len(chopper_frequencies_hz)} chopper frequencies (Hz):", chopper_frequencies_hz)

## Sweep settings

Edit these before each run. The MFLI filter/rate settings are fixed for
the whole sweep (not scaled with chopper frequency) so the SNR comparison
reflects the underlying noise environment at each frequency rather than a
changing measurement bandwidth -- see the discussion in
`measurement/chopper_sweep.py`.

In [ ]:
lasing_voltage = 29.0  # Avtech pulse voltage to hold during measurement
gate_voltage = 5.0  # CH1/CH2 square-wave span, in volts
n_settle_periods = 10  # minimum chopper periods to wait before measuring
ramp_step_size = 1.0  # maximum Avtech voltage change per ramp step, in volts
ramp_sleep_time = 2.0  # seconds to wait after each Avtech ramp step
mfli_time_constant = 0.1  # low-pass filter time constant, in seconds (fixed across the sweep)
mfli_filter_order = 6  # low-pass filter order (fixed across the sweep)
mfli_demod_rate_hz = 100.0  # demodulator output sample rate, in samples/s (fixed across the sweep)
mfli_n_samples = 30  # MFLI samples averaged per chopper frequency
mfli_delay = 0.3  # seconds between successive MFLI samples within one averaged read
max_runtime_s = 10000.0  # safety cutoff: stop the sweep if it runs longer than this, in seconds

print(
    f"lasing_voltage={lasing_voltage}, gate_voltage={gate_voltage}, "
    f"n_settle_periods={n_settle_periods}, ramp_step_size={ramp_step_size}, "
    f"ramp_sleep_time={ramp_sleep_time}, mfli_time_constant={mfli_time_constant}, "
    f"mfli_filter_order={mfli_filter_order}, mfli_demod_rate_hz={mfli_demod_rate_hz}, "
    f"mfli_n_samples={mfli_n_samples}, mfli_delay={mfli_delay}, max_runtime_s={max_runtime_s}"
)

## Live-plot callback

In [ ]:
def live_plot(df_so_far):
    """Redraw the R-vs-frequency plot in place as the sweep progresses."""
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(6, 4))
    plotting.plot_r_vs_frequency(df_so_far, ax=ax)
    plt.tight_layout()
    plt.show()

## Run the sweep and save the result, or fall back to synthetic dummy_data

This is the "one cell" step: it runs the sweep (live-plotting as it goes),
and always ends with `chopper_result` holding data + metadata, whether the
sweep completed, was interrupted, or timed out.

In [ ]:
chopper_result = None

if instruments_connected:
    chopper_config = ChopperSweepConfig(
        lasing_voltage=lasing_voltage,
        gate_voltage=gate_voltage,
        n_settle_periods=n_settle_periods,
        ramp_step_size=ramp_step_size,
        ramp_sleep_time=ramp_sleep_time,
        mfli_time_constant=mfli_time_constant,
        mfli_filter_order=mfli_filter_order,
        mfli_demod_rate_hz=mfli_demod_rate_hz,
        mfli_n_samples=mfli_n_samples,
        mfli_delay=mfli_delay,
        max_runtime_s=max_runtime_s,
    )
    try:
        chopper_result = run_chopper_frequency_sweep(
            avtech,
            qdac,
            mfli,
            frequencies_hz=chopper_frequencies_hz,
            sweep_config=chopper_config,
            on_step=live_plot,
        )
        print(
            f"Sweep finished with status={chopper_result.status!r} "
            f"({chopper_result.completed_points}/{chopper_result.total_points} points)."
        )
    except Exception as exc:
        # Anything other than a plain interrupt (which run_chopper_frequency_sweep
        # already handles internally) -- e.g. a real communication error.
        print(f"Sweep failed unexpectedly: {exc}")

if chopper_result is None:
    # NOTE: hardware not available (or the sweep failed outright) -- this
    # is clearly-labeled synthetic dummy data, NOT a real measurement. It
    # loosely mimics an SNR peak somewhere in the middle of the frequency
    # range (e.g. away from low-frequency 1/f noise and high-frequency
    # bandwidth limits).
    rng = np.random.default_rng(0)
    freqs = np.array(chopper_frequencies_hz, dtype=float)
    peak_freq = 100.0
    snr_shape = np.exp(-0.5 * (np.log(freqs / peak_freq) / 1.0) ** 2)
    lockin_r = 0.02 * snr_shape + 0.002
    lockin_r_std = lockin_r / (5.0 * snr_shape + 1.0) + 1e-5
    lockin_x = lockin_r * 0.9
    lockin_x_std = lockin_r_std * 0.9
    lockin_y = lockin_r * 0.436
    lockin_y_std = lockin_r_std * 0.436
    lockin_phase = np.arctan2(lockin_y, lockin_x)

    dummy_data = pd.DataFrame(
        {
            "chopper_frequency_hz": freqs,
            "lockin_x": lockin_x,
            "lockin_x_std": lockin_x_std,
            "lockin_y": lockin_y,
            "lockin_y_std": lockin_y_std,
            "lockin_r": lockin_r,
            "lockin_r_std": lockin_r_std,
            "lockin_phase": lockin_phase,
        }
    )
    print("Using synthetic dummy_data (NOT a real measurement).")
    now = datetime.now(timezone.utc).isoformat()
    chopper_result = ChopperSweepResult(
        data=dummy_data,
        start_time=now,
        end_time=now,
        status="synthetic_dummy_data",
        completed_points=len(freqs),
        total_points=len(freqs),
        sweep_config=ChopperSweepConfig(),
    )

output_dir = project_root / "data"
chopper_run_dir = data_io.save_sweep_result(chopper_result, output_dir)
print(f"Saved chopper sweep result to {chopper_run_dir} (sweep.csv + metadata.json).")

chopper_result.data

## Reload the saved result to confirm the round-trip

In [ ]:
reloaded_chopper_df, reloaded_chopper_metadata = data_io.load_sweep_result(chopper_run_dir)
print("Reloaded metadata:", reloaded_chopper_metadata)
reloaded_chopper_df

## Plot R vs. chopper frequency

In [ ]:
if chopper_result.status == "synthetic_dummy_data":
    print("NOTE: the plots below are based on synthetic dummy_data, not a real measurement.")
else:
    print(f"The plots below are based on a real sweep measurement (status={chopper_result.status!r}).")

In [ ]:
plotting.plot_r_vs_frequency(chopper_result.data, show=True)

## Plot SNR (R, X, Y) vs. chopper frequency

Pass a subset via `quantities` (e.g. `quantities=("r",)`) to show only
some of the lines.

In [ ]:
plotting.plot_snr_vs_frequency(chopper_result.data, quantities=("r", "x", "y"), show=True)

## Clean up and disconnect all instruments

In [ ]:
try:
    if instruments_connected:
        avtech.output_off()
        qdac.abort_square_wave(1)
        qdac.abort_square_wave(2)
        qdac.set_channel_voltage(1, 0.0)
        qdac.set_channel_voltage(2, 0.0)
        print("Avtech output disabled; QDac channels aborted and zeroed.")
except Exception as exc:
    print(f"Error during instrument cleanup: {exc}")

In [ ]:
for name, instrument in [("QDac", qdac), ("Avtech", avtech), ("MFLI", mfli)]:
    try:
        if instruments_connected:
            instrument.disconnect()
            print(f"Disconnected from {name}.")
    except Exception as exc:
        print(f"Error disconnecting {name}: {exc}")

if not instruments_connected:
    print("Instruments were never connected; nothing to disconnect.")